# 06 – The real test: a second forest, with the rules fixed in advance

In notebook 05 we picked the probability-drop cut-off (0.5) **while looking at Global Forest Watch's answers** for Hasdeo. That makes Hasdeo's score look better than it really is, like choosing your exam strategy after seeing the answer key.

The fix: take the **exact same method and settings**, change nothing, and run it on a **place the model and we have never looked at**. Whatever score comes out is honest.

**The new area: Talabira, Odisha** (about 21.735° N, 83.989° E). A forest near the Hirakud reservoir that was cleared for the **Talabira II & III open-pit coal mine**. Tree felling started in **December 2019**, widely reported in the news. So an image from early 2019 shows the forest *before*, and one from 2025 shows the mine *after*.

**What's different from Hasdeo** (which makes this a tougher test):
- a different state (Odisha, not Chhattisgarh)
- a much more **mixed landscape**: towns, fields, factories, and a big **reservoir** (lots of water, which the model might confuse with dark ground)
- **March** images instead of February. We picked the clearest pair: winter smog makes many images here hazy

**Settings fixed in advance (copied from notebooks 04–05, not changed):**
| Setting | Value |
|---|---|
| Patch size / grid | 64 × 64 px (640 m), 25 × 25 grid = 16 km |
| Model | the same trained `resnet50_eurosat.pth` |
| Method A | Forest (2019) → anything else (2025) |
| Method B | Forest-probability drop ≥ **0.5** |
| GFW "lost" patch | ≥ **10%** of the patch lost in 2019–2024 |

Takes 1–2 minutes. **Run → Run All Cells**.

## Step 1 – Setup

Everything here was explained in notebooks 04 and 05, so this notebook moves faster. The settings in CAPITALS are the fixed rules from the table above.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from PIL import Image

import rasterio
from rasterio.warp import transform as reproject_points, reproject, Resampling
from rasterio.windows import Window
from pystac_client import Client

import torch
import torch.nn as nn
from torchvision import models, transforms

DATA_DIR = Path("C:/data/EuroSAT")
MODEL_PATH = Path("../models/resnet50_eurosat.pth")
FIG_DIR = Path("../figures")
RESULTS_DIR = Path("../results")

LAT, LON = 21.735, 83.989                       # Talabira II & III coal mine, Odisha
SCENES = {2019: "S2B_44QRK_20190324_0_L2A",      # clearest early-2019 image (before felling began, Dec 2019)
          2025: "S2B_44QRK_20250325_0_L2A"}      # clearest early-2025 image, same time of year
GFW_URL = "https://storage.googleapis.com/earthenginepartners-hansen/GFC-2025-v1.13/Hansen_GFC-2025-v1.13_lossyear_30N_080E.tif"

# Fixed rules from Hasdeo, NOT tuned here
N_PATCHES, PATCH = 25, 64
PATCH_KM2 = 0.4096
DROP_CUTOFF = 0.5
GFW_THRESHOLD = 0.10

SIZE = N_PATCHES * PATCH
device = "cuda" if torch.cuda.is_available() else "cpu"
class_names = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
FOREST = class_names.index("Forest")
print("Ready. Device:", device)

## Step 2 – Download the two images

Same as notebook 04: cut a 1,600 × 1,600 pixel (16 km) square around the mine from each image. Both come from tile `44QRK`, so they line up pixel for pixel.

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
images, geo = {}, {}

for year, scene_id in SCENES.items():
    item = catalog.get_collection("sentinel-2-l2a").get_item(scene_id)
    with rasterio.open(item.assets["visual"].href) as src:
        xs, ys = reproject_points("EPSG:4326", src.crs, [LON], [LAT])
        row, col = src.index(xs[0], ys[0])
        window = Window(col - SIZE // 2, row - SIZE // 2, SIZE, SIZE)
        images[year] = np.moveaxis(src.read(window=window), 0, -1)
        geo[year] = (src.crs.to_string(), src.window_transform(window))
    print(f"{year}: {item.properties['datetime'][:10]}, cloud {item.properties['eo:cloud_cover']:.2f}%, blank pixels: {(images[year].sum(-1) == 0).sum()}")

assert geo[2019] == geo[2025], "images must line up"
grid_crs, grid_transform = geo[2019]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, year in zip(axes, SCENES):
    ax.imshow(images[year]); ax.set_title(f"Talabira {year}", fontsize=14); ax.axis("off")
plt.suptitle("Talabira, Odisha – 16 km × 16 km around the Talabira II & III coal mine", fontsize=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "talabira_2019_vs_2025.png", dpi=120)
plt.show()

**Look for:** the dark green forest patch north of the centre in 2019, which has become a pale open-pit mine by 2025. Also note the large reservoir (Hirakud) winding through the west and south, and the factories and towns to the north-east.

## Step 3 – Classify every patch, both years

Exactly as notebook 04: cut into 625 patches, run the model, keep the class, confidence and Forest probability.

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device).eval()
prepare = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def to_patches(img):
    return img.reshape(N_PATCHES, PATCH, N_PATCHES, PATCH, 3).swapaxes(1, 2).reshape(-1, PATCH, PATCH, 3)

pred, conf, forest_prob = {}, {}, {}
for year in SCENES:
    batch = torch.stack([prepare(Image.fromarray(p)) for p in to_patches(images[year])])
    with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16):
        probs = torch.cat([torch.softmax(model(b.to(device)).float(), dim=1) for b in batch.split(128)]).cpu().numpy()
    pred[year] = probs.argmax(1).reshape(N_PATCHES, N_PATCHES)
    conf[year] = probs.max(1).reshape(N_PATCHES, N_PATCHES)
    forest_prob[year] = probs[:, FOREST].reshape(N_PATCHES, N_PATCHES)
    counts = np.bincount(pred[year].ravel(), minlength=10)
    top = ", ".join(f"{class_names[i]} {counts[i]}" for i in np.argsort(-counts)[:4])
    print(f"{year}: average confidence {conf[year].mean():.1%} | most common: {top}")

In [ ]:
colours = plt.get_cmap("tab10")
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for row, year in enumerate(SCENES):
    axes[row, 0].imshow(images[year]); axes[row, 0].set_title(f"{year} – satellite image")
    axes[row, 1].imshow(pred[year], cmap=colours, vmin=-0.5, vmax=9.5, interpolation="nearest")
    axes[row, 1].set_title(f"{year} – model's land cover map")
for ax in axes.flat:
    ax.axis("off")
fig.legend(handles=[Patch(color=colours(i), label=n) for i, n in enumerate(class_names)], loc="lower center", ncol=5)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(FIG_DIR / "talabira_landcover_maps.png", dpi=100)
plt.show()

Compare the maps with the images. Is the reservoir called **SeaLake** or **River**? Are the towns **Residential** or **Industrial**? This tells you how well the model copes with this new landscape, before we even look at deforestation.

## Step 4 – Global Forest Watch's answer for this area

Same as notebook 05: re-draw GFW's loss-year map onto our grid, then work out what fraction of each patch lost trees in **2019–2024** (between our two images).

In [ ]:
gfw_year = np.zeros((SIZE, SIZE), dtype=np.uint8)
with rasterio.open(GFW_URL) as src:
    reproject(rasterio.band(src, 1), gfw_year, dst_transform=grid_transform, dst_crs=grid_crs, resampling=Resampling.nearest)

for y in range(2017, 2026):
    print(f"  {y}: {(gfw_year == y - 2000).sum() / 10_000:5.2f} km²")

lost_pixels = (gfw_year >= 19) & (gfw_year <= 24)
gfw_fraction = lost_pixels.reshape(N_PATCHES, PATCH, N_PATCHES, PATCH).mean(axis=(1, 3))
gfw_loss = gfw_fraction >= GFW_THRESHOLD
print(f"Total GFW loss 2019–2024: {lost_pixels.sum() / 10_000:.2f} km², in {gfw_loss.sum()} patches (≥{GFW_THRESHOLD:.0%})")

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(images[2025])
ax.imshow(np.ma.masked_where(~lost_pixels, gfw_year.astype(int) + 2000), cmap="autumn_r", vmin=2019, vmax=2024, interpolation="nearest")
ax.set_title("GFW tree cover loss 2019–2024 on the 2025 image")
ax.axis("off")
plt.show()

## Step 5 – Score both methods, with the fixed settings

Nothing is tuned here. Whatever comes out is the honest result.

In [ ]:
def score(detected, truth):
    tp, fp, fn = (detected & truth).sum(), (detected & ~truth).sum(), (~detected & truth).sum()
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return dict(TP=tp, FP=fp, FN=fn, precision=precision, recall=recall, f1=f1)

method_a = (pred[2019] == FOREST) & (pred[2025] != FOREST)
method_b = (forest_prob[2019] - forest_prob[2025]) >= DROP_CUTOFF
talabira = {"A: label change": score(method_a, gfw_loss), f"B: prob drop ≥ {DROP_CUTOFF}": score(method_b, gfw_loss)}

print(f"Forest patches in 2019 (model): {(pred[2019] == FOREST).sum()}\n")
print(f"{'Method':24} {'alarms':>7} {'TP':>4} {'FP':>4} {'FN':>4} {'precision':>10} {'recall':>7} {'F1':>5}")
for name, s in talabira.items():
    print(f"{name:24} {s['TP'] + s['FP']:7d} {s['TP']:4d} {s['FP']:4d} {s['FN']:4d} {s['precision']:10.0%} {s['recall']:7.0%} {s['f1']:5.0%}")

In [ ]:
def draw(ax, mask, colour):
    for r, c in zip(*np.nonzero(mask)):
        ax.add_patch(Rectangle((c * PATCH, r * PATCH), PATCH, PATCH, fill=False, edgecolor=colour, lw=2.2))

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
for ax, (name, detected) in zip(axes, [("A: label change", method_a), (f"B: prob drop ≥ {DROP_CUTOFF}", method_b)]):
    s = talabira[name]
    ax.imshow(images[2025])
    draw(ax, detected & gfw_loss, "lime")
    draw(ax, detected & ~gfw_loss, "red")
    draw(ax, ~detected & gfw_loss, "yellow")
    ax.set_title(f"Talabira – {name} (P {s['precision']:.0%}, R {s['recall']:.0%})", fontsize=13)
    ax.axis("off")
axes[1].legend(handles=[Patch(edgecolor=c, fill=False, label=l) for c, l in
                        [("lime", "Both found loss"), ("red", "Our false alarm"), ("yellow", "We missed it")]],
               loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "talabira_validation_map.png", dpi=110)
plt.show()

**What we found when we checked this run by eye:**
- ✅ The **green** boxes sit on the new open pit, exactly where GFW also recorded loss.
- ❌ Two **red** boxes in the east sit under **small clouds and their shadows** in the 2025 image. The scene is 0.55% cloudy overall, and some of it landed here. A new kind of false alarm that Hasdeo didn't have.
- ❓ The red box in the south sits on a **freshly cleared pale rectangle**: something really changed, but it probably wasn't tree cover, so GFW doesn't count it. Our "Forest → not Forest" rule can't tell the difference.
- ❓ The red boxes next to the mine are on its older, dark coal areas and edges. Partly real change, partly mixed patches.
- 📅 GFW puts most loss here in **2023–2024**, not right after the December 2019 felling reported in the news. Either the main clearing at this spot came later, or GFW detected it late. Worth a sentence in a write-up: even the "ground truth" has timing uncertainty.

## Step 6 – Hasdeo vs Talabira, side by side

The Hasdeo numbers are copied from notebook 05 (same settings). Remember: Hasdeo's method B score is **flattering** (cut-off picked using its answers). Talabira's is **honest** (cut-off fixed before we looked).

In [ ]:
hasdeo = {"A: label change": dict(precision=0.47, recall=0.82, f1=0.60),
          f"B: prob drop ≥ {DROP_CUTOFF}": dict(precision=0.53, recall=0.82, f1=0.64)}

print(f"{'Method':24} {'Area':10} {'precision':>10} {'recall':>7} {'F1':>5}")
for name in talabira:
    for area, s in (("Hasdeo", hasdeo[name]), ("Talabira", talabira[name])):
        print(f"{name:24} {area:10} {s['precision']:10.0%} {s['recall']:7.0%} {s['f1']:5.0%}")
    print()

x = np.arange(2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, metric in zip(axes, ("precision", "recall")):
    for k, (area, data, colour) in enumerate((("Hasdeo (tuned)", hasdeo, "lightgray"), ("Talabira (honest)", talabira, "seagreen"))):
        vals = [data[n][metric] * 100 for n in talabira]
        bars = ax.bar(x + (k - 0.5) * 0.38, vals, 0.38, label=area, color=colour)
        ax.bar_label(bars, fmt="%.0f%%", fontsize=9)
    ax.set_xticks(x, ["A: label change", "B: prob drop"])
    ax.set_title(metric.capitalize()); ax.set_ylim(0, 110)
axes[0].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "hasdeo_vs_talabira.png", dpi=150)
plt.show()

## Step 7 – Save the results

In [ ]:
np.savez_compressed(
    RESULTS_DIR / "talabira_predictions.npz",
    pred_2019=pred[2019], pred_2025=pred[2025], conf_2019=conf[2019], conf_2025=conf[2025],
    forest_prob_2019=forest_prob[2019], forest_prob_2025=forest_prob[2025], gfw_fraction=gfw_fraction,
    crs=grid_crs, transform=np.array(tuple(grid_transform)[:6]), class_names=np.array(class_names),
    scenes=np.array(list(SCENES.values())),
)
print("Saved results/talabira_predictions.npz")

## Think about it (answer in chat)

1. Did the method hold up on the new area? Compare precision and recall with Hasdeo.
2. Did method B (probability drop) still beat method A here, now that the cut-off wasn't tuned on this area?
3. Look at the red boxes (false alarms) at Talabira. Are they the same kind of mistake as Hasdeo's (hill shadows), or something new?
4. Why is a result on a place you didn't tune on worth more than a high score on the place you did?